<a href="https://colab.research.google.com/github/olaidczak/dl-project-1/blob/twardowski_branch/dl_project_version7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
!pip uninstall -y sympy
!pip install sympy==1.13.3
!pip install timm -q
import timm
from torch import nn
from torch.utils.data import Subset, DataLoader
from torchvision import datasets, transforms, models
import torch.optim as optim
import requests
import random
import kagglehub
import datetime
import os
import json
import numpy as np
from datetime import datetime
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from collections import defaultdict
from timeit import default_timer as timer
from google.colab import runtime


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
seed = 134
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
# Deterministyczne wyniki GPU (fix #3)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device


'cuda'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
image_path = Path(kagglehub.dataset_download("mengcius/cinic10"))
train_dir = image_path / "train"
valid_dir = image_path / "valid"
test_dir = image_path / "test"

100%|██████████| 754M/754M [00:47<00:00, 16.8MB/s]

Extracting files...


In [ ]:
# Statystyki CINIC-10
CINIC_MEAN = [0.47889522, 0.47227842, 0.43047404]
CINIC_STD  = [0.24205776, 0.23828046, 0.25874835]

# ── SaliencyMix (implementacja wg oryginalnej papierki) ───────────────────────
def saliency_bbox(img, lam):
    """
    Wyznacza prostokąt wycięcia na podstawie mapy saliency obrazu.
    img  : tensor (C, H, W) — pojedynczy obraz z batcha
    lam  : float — współczynnik mieszania z rozkładu Beta
    Zwraca: (bbx1, bby1, bbx2, bby2)
    """
    W = img.size(2)
    H = img.size(1)

    cut_rat = np.sqrt(1.0 - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    # Mapa saliency: suma gradientów po kanałach (gradient obrazu jako proxy)
    sal = img.abs().sum(dim=0)          # (H, W) — jasność jako uproszczone saliency
    # Znajdź piksel o największej saliency
    max_idx = sal.view(-1).argmax().item()
    cy = max_idx // W
    cx = max_idx % W

    bbx1 = max(cx - cut_w // 2, 0)
    bby1 = max(cy - cut_h // 2, 0)
    bbx2 = min(cx + cut_w // 2, W)
    bby2 = min(cy + cut_h // 2, H)

    return bbx1, bby1, bbx2, bby2


def saliency_mix(images, labels, alpha=0.5):
    """
    SaliencyMix augmentation na całym batchu.
    Zwraca: (mixed_images, labels_a, labels_b, lam)
    """
    lam = np.random.beta(alpha, alpha)
    batch_size = images.size(0)
    rand_index = torch.randperm(batch_size).to(images.device)

    labels_a = labels
    labels_b = labels[rand_index]
    mixed = images.clone()

    for i in range(batch_size):
        bbx1, bby1, bbx2, bby2 = saliency_bbox(images[rand_index[i]], lam)
        mixed[i, :, bby1:bby2, bbx1:bbx2] = images[rand_index[i], :, bby1:bby2, bbx1:bbx2]

    # Rzeczywisty lam po wycięciu
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) /
                (images.size(2) * images.size(3)))
    return mixed, labels_a, labels_b, lam


# ── Fabryka transformów ───────────────────────────────────────────────────────
def build_transforms(data_aug: str):
    """
    Zwraca (train_transform, eval_transform).

    data_aug:
        'none'        — tylko Normalize (brak augmentacji)
        'standard'    — RandomCrop + HorizontalFlip + Normalize
        'saliencymix' — j.w. + SaliencyMix aplikowany w pętli treningowej
    """
    eval_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=CINIC_MEAN, std=CINIC_STD)
    ])

    if data_aug == 'none':
        train_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=CINIC_MEAN, std=CINIC_STD)
        ])
    elif data_aug in ('standard', 'saliencymix'):
        train_transform = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean=CINIC_MEAN, std=CINIC_STD)
        ])
    else:
        raise ValueError(f"Nieznana augmentacja: '{data_aug}'. Użyj: 'none', 'standard', 'saliencymix'")

    return train_transform, eval_transform


In [ ]:
def build_resnet18(num_classes=10, weights="IMAGENET1K_V1", dropout=0):
  model = models.resnet18(weights=weights)
  # zmiany ktore pozwalją lepiej dzialac na obrazkach 32x32, bo defaultowo sa 224x224
  model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
  model.maxpool = nn.Identity()

  # Add dropout
  model.fc = nn.Sequential(
    nn.Dropout(dropout),
    nn.Linear(model.fc.in_features, num_classes)
)
  return model

In [ ]:
def build_densenet121(num_classes=10, weights="IMAGENET1K_V1", dropout=0):
    model = models.densenet121(weights=weights)

    # Adjust for 32x32 images
    model.features.conv0 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.features.pool0 = nn.Identity()

    num_ftrs = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(num_ftrs, num_classes)
    )
    return model


def build_ghostnetv3(num_classes=10, dropout=0):
    """
    GhostNetV3 z wagami pretrained ImageNet (przez timm).

    num_classes=0 usuwa oryginalną głowicę timm (trenowaną na 1000 klas ImageNet),
    żebyśmy mogli dołączyć własną głowicę na 10 klas z kontrolowanym dropoutem.
    Backbone i jego wagi ImageNet pozostają nienaruszone.
    """
    backbone = timm.create_model('ghostnetv3_100', pretrained=True, num_classes=0)
    num_ftrs = backbone.num_features

    # Własna głowica: pooling → flatten → dropout → linear(10)
    backbone.head = nn.Sequential(
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Dropout(dropout),
        nn.Linear(num_ftrs, num_classes)
    )
    return backbone


In [ ]:
def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               data_aug: str = 'standard'):
    model.train()
    train_loss, train_acc = 0, 0

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        if data_aug == 'saliencymix':
            X, y_a, y_b, lam = saliency_mix(X, y)
            y_pred = model(X)
            loss = lam * loss_fn(y_pred, y_a) + (1 - lam) * loss_fn(y_pred, y_b)
        else:
            y_pred = model(X)
            loss = loss_fn(y_pred, y)

        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item() / len(y_pred)

    train_loss = train_loss / len(dataloader)
    train_acc  = train_acc  / len(dataloader)
    return train_loss, train_acc


def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module):
    model.eval()
    test_loss, test_acc = 0, 0

    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            test_pred_logits = model(X)
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()
            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += ((test_pred_labels == y).sum().item() / len(test_pred_labels))

    test_loss = test_loss / len(dataloader)
    test_acc  = test_acc  / len(dataloader)
    return test_loss, test_acc


In [ ]:
import copy
import csv
import os

# ── Checkpoint: zapis ─────────────────────────────────────────────────────────
def save_checkpoint(epoch, model, optimizer, scheduler, results,
                    best_acc, best_weights, path,
                    curves_path=None, timestamp=None):
    """
    Zapisuje pełny stan trenowania do .ckpt na Google Drive.
    Opcjonalnie dopisuje nowe epoki do curves.csv.
    """
    checkpoint = {
        "epoch":              epoch,
        "model_state":        model.state_dict(),
        "optimizer_state":    optimizer.state_dict(),
        "scheduler_state":    scheduler.state_dict() if scheduler is not None else None,
        "results":            results,
        "best_acc_test":      best_acc,
        "best_model_weights": best_weights,
    }
    torch.save(checkpoint, path)
    print(f"  [Checkpoint] Zapisano epoka {epoch+1} -> {path}")

    if curves_path is not None and timestamp is not None:
        curves_header = ["timestamp", "epoch", "train_loss", "train_acc", "val_loss", "val_acc"]
        curves_exists = os.path.isfile(curves_path)
        already_saved = 0
        if curves_exists:
            with open(curves_path, "r") as f:
                already_saved = sum(1 for row in csv.DictReader(f)
                                    if str(row["timestamp"]) == str(timestamp))
        with open(curves_path, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=curves_header)
            if not curves_exists:
                writer.writeheader()
            for ep in range(already_saved, epoch + 1):
                writer.writerow({
                    "timestamp":  timestamp,
                    "epoch":      ep + 1,
                    "train_loss": round(results["train_loss"][ep], 6),
                    "train_acc":  round(results["train_acc"][ep],  6),
                    "val_loss":   round(results["test_loss"][ep],  6),
                    "val_acc":    round(results["test_acc"][ep],   6),
                })
        print(f"  [Checkpoint] Zaktualizowano curves.csv (epoki {already_saved+1}-{epoch+1})")


# ── Checkpoint: wczytanie ─────────────────────────────────────────────────────
def load_checkpoint(path, model, optimizer, scheduler):
    """
    Wczytuje checkpoint i przywraca stan modelu/optimizera/schedulera.
    Zwraca: (start_epoch, results, best_acc, best_weights)
    """
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    if scheduler is not None and ckpt["scheduler_state"] is not None:
        scheduler.load_state_dict(ckpt["scheduler_state"])
    print(f"  [Checkpoint] Wznowienie od epoki {ckpt['epoch']+2} "
          f"(ostatnia zapisana: {ckpt['epoch']+1})")
    return ckpt["epoch"] + 1, ckpt["results"], ckpt["best_acc_test"], ckpt["best_model_weights"]


# ── Główna funkcja treningowa ─────────────────────────────────────────────────
def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          save_dir: str,
          scheduler=None,
          data_aug: str = 'standard',
          checkpoint_every: int = 5,
          model_name: str = 'model',
          resume_from: str = None,
          curves_path: str = None,
          timestamp: int = None):
    """
    Trenuje model z checkpointingiem co checkpoint_every epok.

    resume_from : ścieżka do .ckpt — wznowienie przerwanego treningu
                  None — trening od zera
    """
    results = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
    best_acc_test = 0
    best_model_weights = copy.deepcopy(model.state_dict())
    start_epoch = 0

    # Wznowienie
    if resume_from is not None and os.path.isfile(resume_from):
        start_epoch, results, best_acc_test, best_model_weights = load_checkpoint(
            resume_from, model, optimizer, scheduler
        )
    elif resume_from is not None:
        print(f"  [Checkpoint] Nie znaleziono {resume_from}, trening od zera.")

    checkpoint_path = os.path.join(save_dir, "checkpoint_latest.ckpt")

    for epoch in tqdm(range(start_epoch, epochs)):
        train_loss, train_acc = train_step(
            model=model, dataloader=train_dataloader,
            loss_fn=loss_fn, optimizer=optimizer, data_aug=data_aug
        )
        test_loss, test_acc = test_step(
            model=model, dataloader=test_dataloader, loss_fn=loss_fn
        )

        if scheduler is not None:
            scheduler.step()

        if test_acc > best_acc_test:
            best_acc_test = test_acc
            best_model_weights = copy.deepcopy(model.state_dict())

        print(
            f"Epoch: {epoch+1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"valid_loss: {test_loss:.4f} | "
            f"valid_acc: {test_acc:.4f}"
        )

        results["train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
        results["train_acc"].append(train_acc.item()   if isinstance(train_acc,  torch.Tensor) else train_acc)
        results["test_loss"].append(test_loss.item()   if isinstance(test_loss,  torch.Tensor) else test_loss)
        results["test_acc"].append(test_acc.item()    if isinstance(test_acc,   torch.Tensor) else test_acc)

        # Checkpoint co checkpoint_every epok
        if (epoch + 1) % checkpoint_every == 0:
            save_checkpoint(
                epoch, model, optimizer, scheduler,
                results, best_acc_test, best_model_weights,
                checkpoint_path,
                curves_path=curves_path,
                timestamp=timestamp,
            )

    # Koniec treningu
    model.load_state_dict(best_model_weights)
    final_model_path = os.path.join(save_dir, f"{model_name}_{timestamp}.pt")
    torch.save(best_model_weights, final_model_path)
    print(f"  [Trening zakończony] Najlepszy model -> {final_model_path}")

    if os.path.isfile(checkpoint_path):
        os.remove(checkpoint_path)
        print(f"  [Checkpoint] Usunięto tymczasowy checkpoint.")

    results["best_val_acc"] = best_acc_test
    return results


# ── Zapis wyników do CSV ──────────────────────────────────────────────────────
def save_results_to_csv(results, config, curves_path, experiments_path):
    """
    curves_path      — krzywe uczenia (jedna linia per epoka)
    experiments_path — podsumowanie eksperymentu (jedna linia per run)
    Pomija epoki już zapisane przez checkpointy.
    """
    timestamp  = config["timestamp"]
    num_epochs = len(results["train_loss"])

    # curves.csv
    curves_header = ["timestamp", "epoch", "train_loss", "train_acc", "val_loss", "val_acc"]
    curves_exists = os.path.isfile(curves_path)
    already_saved = 0
    if curves_exists:
        with open(curves_path, "r") as f:
            already_saved = sum(1 for row in csv.DictReader(f)
                                if str(row["timestamp"]) == str(timestamp))
    with open(curves_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=curves_header)
        if not curves_exists:
            writer.writeheader()
        for epoch in range(already_saved, num_epochs):
            writer.writerow({
                "timestamp":  timestamp,
                "epoch":      epoch + 1,
                "train_loss": round(results["train_loss"][epoch], 6),
                "train_acc":  round(results["train_acc"][epoch],  6),
                "val_loss":   round(results["test_loss"][epoch],  6),
                "val_acc":    round(results["test_acc"][epoch],   6),
            })

    # experiments.csv
    exp_header = [
        "timestamp", "model", "pretrained_weights", "optimizer",
        "learning_rate", "batch_size", "num_epochs", "weight_decay",
        "dropout", "scheduler", "data_augmentation",
        "best_val_acc", "training_time_s"
    ]
    exp_exists = os.path.isfile(experiments_path)
    with open(experiments_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=exp_header)
        if not exp_exists:
            writer.writeheader()
        writer.writerow({
            "timestamp":          timestamp,
            "model":              config["model"],
            "pretrained_weights": config["pretrained_weights"],
            "optimizer":          config["optimizer"],
            "learning_rate":      config["learning_rate"],
            "batch_size":         config["batch_size"],
            "num_epochs":         num_epochs,
            "weight_decay":       config["weight_decay"],
            "dropout":            config["dropout"],
            "scheduler":          config["scheduler"],
            "data_augmentation":  config["data_augmentation"],
            "best_val_acc":       round(results["best_val_acc"], 6),
            "training_time_s":    round(config["training_time"], 2),
        })

    print(f"Zapisano krzywe  -> {curves_path}")
    print(f"Zapisano exp     -> {experiments_path}")


In [ ]:
# ── Ustawienia danych ─────────────────────────────────────────────────────────
# DATA_AUG steruje augmentacją treningową:
#   'none'        — tylko Normalize
#   'standard'    — RandomCrop + HorizontalFlip + Normalize
#   'saliencymix' — standard + SaliencyMix w pętli treningowej
DATA_AUG    = 'standard'
BATCH_SIZE  = 128
NUM_WORKERS = 2

train_transform, eval_transform = build_transforms(DATA_AUG)

train_data = datasets.ImageFolder(root=train_dir, transform=train_transform)
valid_data = datasets.ImageFolder(root=valid_dir, transform=eval_transform)
test_data  = datasets.ImageFolder(root=test_dir,  transform=eval_transform)

class_indices_train = defaultdict(list)
class_indices_valid = defaultdict(list)

for i, label in enumerate(train_data.targets):
    class_indices_train[label].append(i)

for i, label in enumerate(valid_data.targets):
    class_indices_valid[label].append(i)

selected_train, selected_valid = [], []
for label, inds in class_indices_train.items():
    random.shuffle(inds)
    selected_train.extend(inds[:int(0.75 * len(inds))])

for label, inds in class_indices_valid.items():
    random.shuffle(inds)
    selected_valid.extend(inds[:int(0.75 * len(inds))])

subset_train = Subset(train_data, selected_train)
subset_valid = Subset(valid_data, selected_valid)

train_dataloader = DataLoader(subset_train, batch_size=BATCH_SIZE,
                              num_workers=NUM_WORKERS, shuffle=True)
valid_dataloader = DataLoader(subset_valid, batch_size=BATCH_SIZE,
                              num_workers=NUM_WORKERS, shuffle=False)

print(f"Augmentacja: {DATA_AUG} | Batch: {BATCH_SIZE} | "
      f"Train samples: {len(subset_train)} | Valid samples: {len(subset_valid)}")


In [ ]:
# ── Konfiguracja eksperymentu ─────────────────────────────────────────────────
# MODEL:   'resnet18' | 'densenet121' | 'ghostnetv3'
# DATA_AUG musi być zgodny z tym co ustawiono w Cell 9!
MODEL              = 'densenet121'
NUM_EPOCHS         = 40
LEARNING_RATE      = 0.01
WEIGHT_DECAY       = 1e-4
DROPOUT            = 0
PRETRAINED_WEIGHTS = 'IMAGENET1K_V1'   # dla ghostnetv3 wpisz 'timm/ImageNet'

SAVE_DIR = '/content/drive/MyDrive/data'
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)
    print(f'Utworzono folder: {SAVE_DIR}')

CURVES_CSV      = f'{SAVE_DIR}/curves.csv'
EXPERIMENTS_CSV = f'{SAVE_DIR}/experiments.csv'

# Wznowienie: None = od zera | ścieżka .ckpt = kontynuacja
RESUME_FROM = None
# RESUME_FROM = f'{SAVE_DIR}/checkpoint_latest.ckpt'

# ── Budowa modelu ─────────────────────────────────────────────────────────────
if MODEL == 'resnet18':
    model = build_resnet18(10, weights=PRETRAINED_WEIGHTS, dropout=DROPOUT).to(device)
elif MODEL == 'densenet121':
    model = build_densenet121(10, weights=PRETRAINED_WEIGHTS, dropout=DROPOUT).to(device)
elif MODEL == 'ghostnetv3':
    model = build_ghostnetv3(10, dropout=DROPOUT).to(device)
else:
    raise ValueError(f'Nieznany model: {MODEL}')

loss_fn   = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), momentum=0.9,
                      lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Scheduler opcjonalny
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
# SCHEDULER_NAME = 'CosineAnnealingLR'
scheduler      = None
SCHEDULER_NAME = 'None'

timestamp = int(datetime.now().timestamp())

# ── Trening ───────────────────────────────────────────────────────────────────
start_time = timer()
model_results = train(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=valid_dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=NUM_EPOCHS,
    save_dir=SAVE_DIR,
    scheduler=scheduler,
    data_aug=DATA_AUG,             # pobierane z Cell 9
    checkpoint_every=5,
    model_name=MODEL,
    resume_from=RESUME_FROM,
    curves_path=CURVES_CSV,
    timestamp=timestamp,
)
end_time = timer()
print(f'Total training time: {end_time - start_time:.3f} seconds')

# ── Zapis wyników ─────────────────────────────────────────────────────────────
config = {
    'timestamp':          timestamp,
    'model':              MODEL,
    'pretrained_weights': PRETRAINED_WEIGHTS,
    'optimizer':          'SGD',
    'learning_rate':      LEARNING_RATE,
    'batch_size':         BATCH_SIZE,
    'weight_decay':       WEIGHT_DECAY,
    'dropout':            DROPOUT,
    'scheduler':          SCHEDULER_NAME,
    'data_augmentation':  DATA_AUG,
    'training_time':      end_time - start_time,
}

save_results_to_csv(
    results=model_results,
    config=config,
    curves_path=CURVES_CSV,
    experiments_path=EXPERIMENTS_CSV
)
runtime.unassign()
